# 5. 对抗性攻击算法之AutoAttack

## 5.0 上节内容回顾与本节主要内容介绍

在之前的实验中，我们学习了几种白盒攻击算法以及迁移攻击算法。在本节中，我们将简单了解并学习使用一个更强力攻击算法：AutoAttack。

本实验的主要内容为采用Python、PyTorch等技术，学习使用autoattack库实施AutoAttack攻击。

## 5.1 AutoAttack简单介绍

### 5.1.0 简介

AutoAttack由三个白盒攻击以及一个黑盒攻击组成。

白盒攻击：
- APGD-CE：PGD攻击的改良版，在迭代中使用自适应的步长
- APGD-DLR：除了使用自适应的步长之外，使用防止梯度遮蔽的DLR攻击损失
$$ DLR(x, y) = -\frac{z_y - z_{\pi_2}}{z_y - z_{\pi_3}},\, 其中z为logits,z_{\pi_i}为z中元素从大到小排序的第i个元素 $$
- FAB：一种白盒攻击算法，参考：[Fast Adaptive Boundary Attack](https://arxiv.org/abs/1907.02044)<sup>[1]</sup>

黑盒攻击：
- Square：Square攻击是基于分数的黑盒攻击，它使用随机搜索策略并且不利用任何梯度近似。它在查询效率和成功率方面优于其他黑盒攻击，并且已被证明其攻击强度甚至可以与白盒攻击相当。参考：[Square Attack](https://arxiv.org/abs/1912.00049)<sup>[2]</sup>

这些多样性的攻击使得AutoAttack具有相对全面的攻击能力。除此之外，AutoAttack被设计为无参数的攻击算法，能够在相对可负担的计算消耗下实现强力的对抗性攻击。

参考：[AutoAttack](https://github.com/fra31/auto-attack)<sup>[3]</sup>

参考文献：

[1] [Croce F ,  Hein M . Minimally distorted Adversarial Examples with a Fast Adaptive Boundary Attack[C], 2019.](https://arxiv.org/abs/1907.02044)

[2] [Andriushchenko M, Croce F, Flammarion N, et al. Square attack: a query-efficient black-box adversarial attack via random search[C]. European Conference on Computer Vision. Springer, Cham, 2020: 484-501.](https://arxiv.org/abs/1912.00049)

[3] [Croce F, Hein M. Reliable evaluation of adversarial robustness with an ensemble of diverse parameter-free attacks[C]. International conference on machine learning. PMLR, 2020: 2206-2216.](https://github.com/fra31/auto-attack)

### 5.1.1 autoattack库安装

#### 安装方式1：

命令行运行 `pip install git+https://github.com/fra31/auto-attack`

#### 安装方式2：

此方法需要先安装git，安装完成后，命令行运行 `git clone https://github.com/fra31/auto-attack.git` 即可下载autoattack的源代码，之后仅可以在本文件夹下调用autoattack库。

#### 安装方式3：

从[https://github.com/fra31/auto-attack](https://github.com/fra31/auto-attack)下载源代码，上传到此环境，然后解压。之后仅可以在本文件夹下调用autoattack库。

#### 安装方式4:

在3、4的基础上，进入autoattack源代码的文件夹中，在命令行中依此运行：
1.  `python setup.py build`
2.  `python setup.py install`

即可完成autoattack库的安装。

### 5.1.2 导入autoattack库

在安装完成以后，可以在Python中用以下代码导入autoattack

In [2]:
import sys
sys.path.insert(0, r'D:\软件\对抗性防御\对抗性防御-1\03.代码')
import importlib.util
from autoattack import AutoAttack

## 5.2 相关模块导入

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import os
import logging

from tabulate import tabulate
import test; test_fn = test.test
from loss import LabelSmoothingCrossEntropyLoss, CWLoss
from pgd import LinfPGD
from utils import load_mnist_test
from models import LeNet5, FCNet


logger = logging.getLogger('base')
logger.setLevel(logging.DEBUG)

formatter = logging.Formatter(fmt='%(process)6d %(asctime)s %(message)s', datefmt='%Y%m%d %H:%M:%S')
stream_handler = logging.StreamHandler()
stream_handler.setLevel(logging.DEBUG)
stream_handler.setFormatter(formatter)

logger.addHandler(stream_handler)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 5.3 使用AutoAttack进行攻击

In [4]:
# 加载样本
x, y = load_mnist_test()

### 5.3.0 使用AutoAttack攻击LeNet5

In [5]:
std_state = torch.load('./save_model/50epoch/mnist_lenet5.pth')
std_lenet = LeNet5()
std_lenet.load_state_dict(std_state['net'])
std_lenet = std_lenet.to(device)
std_lenet.eval()

LeNet5(
  (conv1): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Sequential(
    (0): Linear(in_features=400, out_features=120, bias=True)
    (1): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=120, out_features=84, bias=True)
    (1): ReLU()
  )
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [6]:
# 创建AutoAttack攻击
adversary = AutoAttack(std_lenet, eps=0.1)

setting parameters for standard version


In [7]:
"""
    执行AutoAttack测试。run_standard_evaluation依此执行apgd-ce、apgd-t、fab-t和square四轮攻击，
并且每一轮攻击只针对上一轮攻击失败的样本。
    run_standard_evaluation默认使用batch_size 250，在每个batch结束时，会打印该batch的攻击成功个
数。比如：

apgd-ce - 1/40 - 184 out of 250 successfully perturbed

表示当前使用apgd-ce攻击，共40个batch，在第一个batch的250个样本中攻击成功184个。
"""
x_adv = adversary.run_standard_evaluation(x, y)

using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 99.00%
apgd-ce - 1/40 - 192 out of 250 successfully perturbed
apgd-ce - 2/40 - 180 out of 250 successfully perturbed
apgd-ce - 3/40 - 196 out of 250 successfully perturbed
apgd-ce - 4/40 - 182 out of 250 successfully perturbed
apgd-ce - 5/40 - 188 out of 250 successfully perturbed
apgd-ce - 6/40 - 185 out of 250 successfully perturbed
apgd-ce - 7/40 - 189 out of 250 successfully perturbed
apgd-ce - 8/40 - 189 out of 250 successfully perturbed
apgd-ce - 9/40 - 183 out of 250 successfully perturbed
apgd-ce - 10/40 - 197 out of 250 successfully perturbed
apgd-ce - 11/40 - 175 out of 250 successfully perturbed
apgd-ce - 12/40 - 190 out of 250 successfully perturbed
apgd-ce - 13/40 - 182 out of 250 successfully perturbed
apgd-ce - 14/40 - 173 out of 250 successfully perturbed
apgd-ce - 15/40 - 183 out of 250 successfully perturbed
apgd-ce - 16/40 - 188 out of 250 successfully perturbed
apgd-ce - 17/40 - 187 o

作为对比，测试PGD100和CW100

In [8]:
# 定义攻击参数
PGD_kwargs = dict(net=std_lenet, eps=0.1, step=100, step_size=0.025, random_start=True)
CW_kwargs = dict(net=std_lenet, eps=0.1, step=100, step_size=0.025, random_start=True, criterion=CWLoss)


In [9]:
# 定义攻击
PGD100 = LinfPGD(**PGD_kwargs)
CW100 = LinfPGD(**CW_kwargs)

In [11]:
# 执行测试
pgd_acc, _ = test_fn(nn.Sequential(PGD100, std_lenet), x, y, bs=250, mode='attack')
logger.info(f'PGD100: {pgd_acc:.2f}')

cw_acc, _ = test_fn(nn.Sequential(CW100, std_lenet), x, y, bs=250, mode='attack')
logger.info(f'CW100: {cw_acc:.2f}')

 19112 20260207 12:45:38 PGD100: 30.57
 19112 20260207 12:46:10 CW100: 30.57


在PGD100攻击和CW100攻击之下，LeNet5模型的准确率约为30%左右<sup>*</sup>，相比之下，经过AutoAttack攻击的LeNet5模型最终准确率仅有约17%，准确率下降超过10%。这表明AutoAttack是更强力的攻击。
<br>

<br>

<small>*PGD攻击、CW攻击以及AutoAttaack攻击均具有随机性，因此每次测试的结果略有不同</small>

### 5.3.1 使用AutoAttack测试Label Smoothing LeNet5

In [12]:
# 加载模型
ls_state = torch.load('./save_model/50epoch/mnist_lenet5_ls0.7.pth')
ls_lenet = LeNet5()
ls_lenet.load_state_dict(ls_state['net'])
ls_lenet = ls_lenet.to(device)
ls_lenet.eval()

LeNet5(
  (conv1): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (conv2): Sequential(
    (0): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc1): Sequential(
    (0): Linear(in_features=400, out_features=120, bias=True)
    (1): ReLU()
  )
  (fc2): Sequential(
    (0): Linear(in_features=120, out_features=84, bias=True)
    (1): ReLU()
  )
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)

In [13]:
adversary = AutoAttack(ls_lenet, eps=0.1, version='standard')

setting parameters for standard version


In [14]:
x_adv = adversary.run_standard_evaluation(x, y, bs=250)

using standard version including apgd-ce, apgd-t, fab-t, square.
initial accuracy: 99.19%
apgd-ce - 1/40 - 55 out of 250 successfully perturbed
apgd-ce - 2/40 - 39 out of 250 successfully perturbed
apgd-ce - 3/40 - 43 out of 250 successfully perturbed
apgd-ce - 4/40 - 40 out of 250 successfully perturbed
apgd-ce - 5/40 - 34 out of 250 successfully perturbed
apgd-ce - 6/40 - 51 out of 250 successfully perturbed
apgd-ce - 7/40 - 45 out of 250 successfully perturbed
apgd-ce - 8/40 - 47 out of 250 successfully perturbed
apgd-ce - 9/40 - 54 out of 250 successfully perturbed
apgd-ce - 10/40 - 41 out of 250 successfully perturbed
apgd-ce - 11/40 - 41 out of 250 successfully perturbed
apgd-ce - 12/40 - 48 out of 250 successfully perturbed
apgd-ce - 13/40 - 43 out of 250 successfully perturbed
apgd-ce - 14/40 - 35 out of 250 successfully perturbed
apgd-ce - 15/40 - 43 out of 250 successfully perturbed
apgd-ce - 16/40 - 50 out of 250 successfully perturbed
apgd-ce - 17/40 - 28 out of 250 success

In [15]:
# 更新PGD100, CW100参数
PGD_kwargs.update(net=ls_lenet)
CW_kwargs.update(net=ls_lenet)

# 重新创建攻击
PGD100 = LinfPGD(**PGD_kwargs)
CW100 = LinfPGD(**CW_kwargs)

# 执行测试
pgd_acc, _ = test(nn.Sequential(PGD100, ls_lenet), x, y, bs=250, mode='attack')
logger.info(f'PGD100: {pgd_acc:.2f}')

cw_acc, _ = test(nn.Sequential(CW100, ls_lenet), x, y, bs=250, mode='attack')
logger.info(f'CW100: {cw_acc:.2f}')

TypeError: 'module' object is not callable. Did you mean: 'test.test(...)'?

在之前的实验中，我们证明了Label Smoothing能够有效地防御FGSM、PGD和CW攻击。其PGD100的准确率为84.43%；CW100的准确率为78.98%；远远高于不使用Label Smoothing训练的LeNet5模型。

然而，AutoAttack能够将Label Smoothing模型的准确率攻击（降低）到8.47%，这与其在PGD100和CW100测试中的表现差异巨大。

这表明Label Smoothing带来的鲁棒性提升可能是依赖于梯度遮蔽的，因而能够在一定程度上防御PGD、CW攻击。然而，面对能够有效规避梯度遮蔽的AutoAttack，它的防御能力就会大打折扣，甚至不如仅使用干净样本进行训练的标准LeNet5模型。